In [55]:
#%pip install opencv-python
#%pip install numpy
#%pip install matplotlib
#%pip install python-chess
#%pip install pandas
#%pip install tensorflow
#%pip install joblib
#%pip install chessimg2pos

In [56]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import chess
import tempfile
import json
import joblib
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization
from features import build_live_feature_row
from exported_chess_winner_models.subtokenizer_reference import move_string_to_subtokens
from collections import deque
from dataclasses import dataclass
from chessimg2pos import predict_fen

In [57]:
EXPORT_DIR = "exported_chess_winner_models"

with open(os.path.join(EXPORT_DIR, "export_metadata.json"), "r") as f:
    export_metadata = json.load(f)

winner_classes = export_metadata["winner_classes"]
rf_feature_cols = export_metadata["rf_winner_best_features"]
move_feature_cols = export_metadata["move_only_feature_cols"]

rf_model = joblib.load(os.path.join(EXPORT_DIR, "rf_winner_best.pkl"))
extra_scaler = joblib.load(os.path.join(EXPORT_DIR, "extra_scaler.pkl"))


# Load metadata
sequence_length = export_metadata["subtok_seq_len"]
vocab = export_metadata["subtok_vocab"]

subtok_vectorizer = TextVectorization(
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize=None,
    split="whitespace"
)

subtok_vectorizer.set_vocabulary(vocab)


@tf.keras.utils.register_keras_serializable()
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.maxlen = maxlen
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            mask_zero=True
        )
        self.pos_emb = layers.Embedding(
            input_dim=maxlen,
            output_dim=embed_dim
        )

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def compute_mask(self, x, mask=None):
        return self.token_emb.compute_mask(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "maxlen": self.maxlen,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
        })
        return config


@tf.keras.utils.register_keras_serializable()
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)

        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, x, training=False, mask=None):
        attention_mask = None

        if mask is not None:
            attention_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)

        attn_output = self.att(
            x,
            x,
            attention_mask=attention_mask,
            training=training
        )

        attn_output = self.drop1(attn_output, training=training)
        out1 = self.ln1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.drop2(ffn_output, training=training)

        return self.ln2(out1 + ffn_output)

    def compute_mask(self, inputs, mask=None):
        return mask

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate,
        })
        return config

@tf.keras.utils.register_keras_serializable()
class MaskedGlobalAveragePooling1D(layers.Layer):
    def call(self, inputs, mask=None):
        if mask is None:
            return tf.reduce_mean(inputs, axis=1)

        mask = tf.cast(mask, dtype=inputs.dtype)
        mask = tf.expand_dims(mask, axis=-1)

        summed = tf.reduce_sum(inputs * mask, axis=1)
        counts = tf.reduce_sum(mask, axis=1)

        return summed / tf.maximum(counts, 1.0)

    def compute_mask(self, inputs, mask=None):
        return None


@tf.keras.utils.register_keras_serializable()
class AttentionPooling1D(layers.Layer):
    def __init__(self, hidden_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.score_dense = layers.Dense(hidden_dim, activation="tanh")
        self.score_out = layers.Dense(1)

    def call(self, inputs, mask=None):
        scores = self.score_out(self.score_dense(inputs))

        if mask is not None:
            mask = tf.cast(mask[:, :, tf.newaxis], scores.dtype)
            scores = scores + (1.0 - mask) * -1e9

        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * weights, axis=1)

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        config = super().get_config()
        config.update({"hidden_dim": self.hidden_dim})
        return config


@tf.keras.utils.register_keras_serializable()
class PhaseAttentionPooling1D(layers.Layer):
    def __init__(self, hidden_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.full_pool = AttentionPooling1D(hidden_dim)
        self.opening_pool = AttentionPooling1D(hidden_dim)
        self.middle_pool = AttentionPooling1D(hidden_dim)
        self.end_pool = AttentionPooling1D(hidden_dim)

    def call(self, inputs, mask=None):
        seq_len = tf.shape(inputs)[1]
        one_third = tf.maximum(seq_len // 3, 1)
        two_third = tf.maximum((2 * seq_len) // 3, one_third + 1)

        opening_x = inputs[:, :one_third, :]
        middle_x = inputs[:, one_third:two_third, :]
        end_x = inputs[:, two_third:, :]

        opening_mask = mask[:, :one_third] if mask is not None else None
        middle_mask = mask[:, one_third:two_third] if mask is not None else None
        end_mask = mask[:, two_third:] if mask is not None else None

        full_vec = self.full_pool(inputs, mask=mask)
        opening_vec = self.opening_pool(opening_x, mask=opening_mask)
        middle_vec = self.middle_pool(middle_x, mask=middle_mask)
        end_vec = self.end_pool(end_x, mask=end_mask)

        return layers.Concatenate()([full_vec, opening_vec, middle_vec, end_vec])

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        config = super().get_config()
        config.update({"hidden_dim": self.hidden_dim})
        return config


@tf.keras.utils.register_keras_serializable()
class MultiPool1D(layers.Layer):
    def __init__(self, hidden_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.att_pool = AttentionPooling1D(hidden_dim)
        self.avg_pool = MaskedGlobalAveragePooling1D()

    def call(self, inputs, mask=None):
        att_vec = self.att_pool(inputs, mask=mask)
        avg_vec = self.avg_pool(inputs, mask=mask)

        if mask is not None:
            mask_expanded = tf.cast(mask[:, :, tf.newaxis], inputs.dtype)
            masked_inputs = inputs + (1.0 - mask_expanded) * -1e9
            max_vec = tf.reduce_max(masked_inputs, axis=1)
        else:
            max_vec = tf.reduce_max(inputs, axis=1)

        return layers.Concatenate()([att_vec, avg_vec, max_vec])

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        config = super().get_config()
        config.update({"hidden_dim": self.hidden_dim})
        return config

transformer_model = tf.keras.models.load_model(
    os.path.join(EXPORT_DIR, "experimental_gated_hybrid_subtok.keras"),
    custom_objects={
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
        "TransformerBlock": TransformerBlock,
        "MaskedGlobalAveragePooling1D": MaskedGlobalAveragePooling1D,
        "AttentionPooling1D": AttentionPooling1D,
        "PhaseAttentionPooling1D": PhaseAttentionPooling1D,
        "MultiPool1D": MultiPool1D,
    },
    compile=False
)

def predict_winner_from_move_list(move_list):
    live_features = build_live_feature_row(move_list)

    X_rf = live_features.reindex(columns=rf_feature_cols, fill_value=0).astype(float)
    rf_probs = rf_model.predict_proba(X_rf)[0]

    move_string = live_features.loc[0, "moves_clean"]
    subtok_string = move_string_to_subtokens(move_string)

    X_moves = subtok_vectorizer(tf.constant([subtok_string]))

    if isinstance(X_moves, dict):
        X_moves = list(X_moves.values())[0]

    X_extra = live_features.reindex(columns=move_feature_cols, fill_value=0).astype(float)
    X_extra_scaled = extra_scaler.transform(X_extra)

    trans_pred = transformer_model.predict([X_moves, X_extra_scaled], verbose=0)

    if isinstance(trans_pred, dict):
        trans_probs = list(trans_pred.values())[0][0]
    elif isinstance(trans_pred, list):
        trans_probs = trans_pred[0][0]
    else:
        trans_probs = trans_pred[0]

    return {
        "classes": winner_classes,
        "rf_probs": np.asarray(rf_probs, dtype=float),
        "transformer_probs": np.asarray(trans_probs, dtype=float),
        "avg_probs": 0.5 * np.asarray(rf_probs, dtype=float) + 0.5 * np.asarray(trans_probs, dtype=float),
    }

C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\layer.py:427: UserWarning: `build()` was called on layer 'token_and_position_embedding_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\layer.py:427: UserWarning: `build()` was called on layer 'transformer_block_11', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\layer

In [58]:
def show_img(img, title="Image", figsize=(7, 7)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def crop_search_region(frame, top=0.06, bottom=0.04, left=0.03, right=0.03):
    h, w = frame.shape[:2]

    y1 = int(top * h)
    y2 = int((1 - bottom) * h)
    x1 = int(left * w)
    x2 = int((1 - right) * w)

    cropped = frame[y1:y2, x1:x2]
    offset = {"x": x1, "y": y1}

    return cropped, offset


def line_length(line):
    x1, y1, x2, y2 = line
    return np.hypot(x2 - x1, y2 - y1)


def line_angle_deg(line):
    x1, y1, x2, y2 = line
    return np.degrees(np.arctan2(y2 - y1, x2 - x1))


def get_hough_segments(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    h, w = img.shape[:2]

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=70,
        minLineLength=min(h, w) // 5,
        maxLineGap=25
    )

    segments = []

    if lines is None:
        return segments

    for line in lines:
        x1, y1, x2, y2 = line[0]
        seg = (x1, y1, x2, y2)

        if line_length(seg) > min(h, w) * 0.4:
            segments.append(seg)

    return segments


def split_line_families(segments):
    row_lines = []
    col_lines = []

    for seg in segments:
        angle = line_angle_deg(seg)

        if angle < -90:
            angle += 180
        if angle > 90:
            angle -= 180

        if abs(angle) < 35:
            row_lines.append(seg)
        else:
            col_lines.append(seg)

    return row_lines, col_lines


def filter_by_dominant_angle(lines, max_deviation=8):
    if len(lines) == 0:
        return []

    angles = []

    for line in lines:
        angle = line_angle_deg(line)

        if angle < -90:
            angle += 180
        if angle > 90:
            angle -= 180

        angles.append(angle)

    median_angle = np.median(angles)

    filtered = []

    for line, angle in zip(lines, angles):
        if abs(angle - median_angle) <= max_deviation:
            filtered.append(line)

    return filtered


def cluster_lines_by_position(lines, orientation="row", cluster_dist=18):
    if len(lines) == 0:
        return []

    positioned = []

    for seg in lines:
        x1, y1, x2, y2 = seg

        if orientation == "row":
            pos = (y1 + y2) / 2
        else:
            pos = (x1 + x2) / 2

        positioned.append((pos, seg))

    positioned.sort(key=lambda t: t[0])

    clusters = []
    current = [positioned[0]]

    for item in positioned[1:]:
        current_mean = np.mean([pos for pos, _ in current])

        if abs(item[0] - current_mean) < cluster_dist:
            current.append(item)
        else:
            clusters.append(current)
            current = [item]

    clusters.append(current)

    merged = []

    for cluster in clusters:
        segs = [seg for _, seg in cluster]
        best = max(segs, key=line_length)
        merged.append(best)

    return merged


def contour_candidates(search_frame):
    min_area_ratio = 0.03
    max_area_ratio = 1.00

    gray = cv2.cvtColor(search_frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (7, 7), 0)
    edges = cv2.Canny(blur, 50, 150)

    kernel = np.ones((7, 7), np.uint8)
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        closed,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = search_frame.shape[:2]
    frame_area = h * w

    candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < min_area_ratio * frame_area:
            continue

        if area > max_area_ratio * frame_area:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)

        if bw <= 0 or bh <= 0:
            continue

        aspect = bw / bh

        if aspect < 0.75 or aspect > 1.35:
            continue

        candidates.append({
            "contour": cnt,
            "x": x,
            "y": y,
            "w": bw,
            "h": bh,
            "area": area,
            "aspect": aspect
        })

    return candidates, edges, closed


def score_candidate_grid(search_frame, cand):
    x, y, w, h = cand["x"], cand["y"], cand["w"], cand["h"]

    pad_x = int(0.08 * w)
    pad_y = int(0.08 * h)

    s_h, s_w = search_frame.shape[:2]

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(s_w, x + w + pad_x)
    y2 = min(s_h, y + h + pad_y)

    crop = search_frame[y1:y2, x1:x2]

    segments = get_hough_segments(crop)
    row_lines, col_lines = split_line_families(segments)

    row_lines = filter_by_dominant_angle(row_lines, max_deviation=8)
    col_lines = filter_by_dominant_angle(col_lines, max_deviation=8)

    row_lines = cluster_lines_by_position(row_lines, "row", cluster_dist=18)
    col_lines = cluster_lines_by_position(col_lines, "col", cluster_dist=18)

    row_count = len(row_lines)
    col_count = len(col_lines)

    grid_score = min(row_count, 10) * min(col_count, 10)
    aspect_penalty = -abs(cand["aspect"] - 1.0)
    area_score = cand["area"] / (search_frame.shape[0] * search_frame.shape[1])

    score = (
        20 * grid_score +
        3 * row_count +
        3 * col_count +
        100 * area_score +
        35 * aspect_penalty
    )

    return score


def find_best_contour_grid_candidate(search_frame):
    candidates, edges, closed = contour_candidates(search_frame)

    best = None
    best_score = -np.inf

    for cand in candidates:
        score = score_candidate_grid(search_frame, cand)

        if score > best_score:
            best_score = score
            best = cand

    return best, candidates


def expand_box_2d(box, frame_shape, pad=0.02):
    s_h, s_w = frame_shape[:2]

    x, y, w, h = box["x"], box["y"], box["w"], box["h"]

    pad_x = int(pad * w)
    pad_y = int(pad * h)

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(s_w, x + w + pad_x)
    y2 = min(s_h, y + h + pad_y)

    return {
        "x": x1,
        "y": y1,
        "w": x2 - x1,
        "h": y2 - y1
    }


def crop_from_box(frame, box):
    x, y, w, h = box["x"], box["y"], box["w"], box["h"]
    return frame[y:y+h, x:x+w]


def detect_board_contour_grid_2d(frame, debug=False):
    search_frame, offset = crop_search_region(frame)

    best, candidates = find_best_contour_grid_candidate(search_frame)

    if best is None:
        return None

    full_box = {
        "x": best["x"] + offset["x"],
        "y": best["y"] + offset["y"],
        "w": best["w"],
        "h": best["h"]
    }

    final_box = expand_box_2d(full_box, frame.shape, pad=0.00)
    board_crop = crop_from_box(frame, final_box)

    if debug:
        vis = frame.copy()
        x, y, w, h = final_box["x"], final_box["y"], final_box["w"], final_box["h"]
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 3)
        show_img(vis, "Detected 2D Board")
        show_img(board_crop, "Board Crop")

    return {
        "final_box": final_box,
        "board_crop": board_crop,
        "candidates": candidates,
        "best_candidate": best
    }

def prepare_2d_board_image(board_crop, output_size=800):

    if board_crop is None:
        return None

    h, w = board_crop.shape[:2]

    if h <= 0 or w <= 0:
        return None

    board_ready = cv2.resize(board_crop, (output_size, output_size))
    return board_ready


def crop_and_prepare_board(frame, board_box, output_size=800):
    board_crop = crop_from_box(frame, board_box)
    board_ready = prepare_2d_board_image(board_crop, output_size=output_size)
    return board_ready

In [59]:



@dataclass
class TrackerConfig:
    output_size: int = 800
    panel_size: int = 380

    board_detect_every_n_frames: int = 15
    reset_if_board_missing_frames: int = 60

    required_stable_frames: int = 2
    required_confirm_frames: int = 1
    confirm_board_diff_threshold: float = 4.0

    periodic_validation_seconds: float = 0.75
    periodic_cooldown_after_motion: int = 16

    adaptive_window_size: int = 45
    adaptive_warmup_frames: int = 10
    adaptive_start_k: float = 5.0
    adaptive_stop_k: float = 2.0

    unstable_active_square_limit: int = 12
    unstable_score_fraction: float = 0.25

    use_fuzzy_single_move: bool = True
    fuzzy_max_distance: int = 1
    allow_multimove_stitching: bool = True
    max_stitch_depth: int = 4

    failed_stitch_recovery_threshold: int = 3

    show_highlights: bool = True
    show_debug: bool = True


def normalize_chessimg2pos_fen(piece_fen):
    rows = piece_fen.strip().split("/")
    normalized_rows = []

    for row in rows:
        new_row = ""
        empty_count = 0

        for ch in row:
            if ch == "1":
                empty_count += 1
            else:
                if empty_count > 0:
                    new_row += str(empty_count)
                    empty_count = 0
                new_row += ch

        if empty_count > 0:
            new_row += str(empty_count)

        normalized_rows.append(new_row)

    return "/".join(normalized_rows)


def is_valid_board_fen(board_fen):
    try:
        chess.Board(board_fen + " w - - 0 1")
        return True
    except Exception:
        return False


def extract_fen_from_board_image(board_img):
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        temp_path = tmp.name

    try:
        cv2.imwrite(temp_path, board_img)
        raw_fen = predict_fen(temp_path)
        board_fen = normalize_chessimg2pos_fen(raw_fen)

        if not is_valid_board_fen(board_fen):
            return None

        return board_fen

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


def safe_extract_fen_from_board_image(board_img, verbose=False):
    try:
        board_fen = extract_fen_from_board_image(board_img)

        if board_fen is None:
            if verbose:
                print("CNN returned invalid FEN.")
            return None

        return board_fen

    except Exception as e:
        if verbose:
            print("CNN FEN extraction failed:", e)
        return None


STARTING_BOARD_FEN = chess.Board().board_fen()


def board_fen_square_distance(fen_a, fen_b):
    board_a = chess.Board(fen_a + " w - - 0 1")
    board_b = chess.Board(fen_b + " w - - 0 1")

    distance = 0

    for sq in chess.SQUARES:
        if board_a.piece_at(sq) != board_b.piece_at(sq):
            distance += 1

    return distance


def create_new_game_state():
    board = chess.Board()

    return {
        "board": board,
        "move_list": [],
        "fen_sequence": [board.board_fen()],
        "last_accepted_fen": board.board_fen(),
        "game_started": False
    }


def should_reset_for_new_game(cnn_board_fen, move_list):
    return cnn_board_fen == STARTING_BOARD_FEN and len(move_list) > 0


def stitch_one_move_from_fen(board, target_board_fen, scores=None):
    matching_moves = []

    for move in board.legal_moves:
        test_board = board.copy()
        test_board.push(move)

        if test_board.board_fen() == target_board_fen:
            matching_moves.append(move)

    if len(matching_moves) == 1:
        return matching_moves[0]

    if len(matching_moves) > 1 and scores is not None:
        best_move = None
        best_score = -1

        for move in matching_moves:
            from_sq = chess.square_name(move.from_square)
            to_sq = chess.square_name(move.to_square)
            score = scores.get(from_sq, 0) + scores.get(to_sq, 0)

            if board.is_castling(move):
                if chess.square_name(move.to_square) in ["g1", "g8"]:
                    rook_from = "h1" if board.turn == chess.WHITE else "h8"
                    rook_to = "f1" if board.turn == chess.WHITE else "f8"
                else:
                    rook_from = "a1" if board.turn == chess.WHITE else "a8"
                    rook_to = "d1" if board.turn == chess.WHITE else "d8"

                score += scores.get(rook_from, 0) + scores.get(rook_to, 0)

            if score > best_score:
                best_score = score
                best_move = move

        return best_move

    return None


def stitch_one_move_from_fen_fuzzy(board, target_board_fen, scores=None, max_distance=1):
    candidates = []

    for move in board.legal_moves:
        test_board = board.copy()
        test_board.push(move)

        distance = board_fen_square_distance(test_board.board_fen(), target_board_fen)

        from_sq = chess.square_name(move.from_square)
        to_sq = chess.square_name(move.to_square)

        score = 0
        if scores is not None:
            score = scores.get(from_sq, 0) + scores.get(to_sq, 0)

        if board.is_castling(move) and scores is not None:
            if chess.square_name(move.to_square) in ["g1", "g8"]:
                rook_from = "h1" if board.turn == chess.WHITE else "h8"
                rook_to = "f1" if board.turn == chess.WHITE else "f8"
            else:
                rook_from = "a1" if board.turn == chess.WHITE else "a8"
                rook_to = "d1" if board.turn == chess.WHITE else "d8"

            score += scores.get(rook_from, 0) + scores.get(rook_to, 0)

        candidates.append((move, distance, score))

    candidates.sort(key=lambda x: (x[1], -x[2]))
    best_move, best_distance, _ = candidates[0]

    if best_distance <= max_distance:
        return best_move, best_distance

    return None, best_distance


def stitch_moves_from_fen(board, target_board_fen, max_depth=4, scores=None, beam_width=500):
    if board.board_fen() == target_board_fen:
        return []

    best_seq = None
    best_cost = float("inf")
    queue = [(board.copy(), [], 0.0)]

    for _ in range(max_depth):
        next_queue = []

        for current_board, move_seq, cost_so_far in queue:
            for move in current_board.legal_moves:
                test_board = current_board.copy()

                from_sq = chess.square_name(move.from_square)
                to_sq = chess.square_name(move.to_square)

                move_score = 0
                if scores is not None:
                    move_score = scores.get(from_sq, 0) + scores.get(to_sq, 0)

                move_cost = 10.0 - move_score

                piece = current_board.piece_at(move.from_square)

                if current_board.is_castling(move):
                    move_cost -= 12.0

                if piece is not None and piece.piece_type == chess.ROOK and move_score <= 0:
                    move_cost += 8.0

                if piece is not None and piece.piece_type == chess.KING and move_score <= 0:
                    move_cost += 6.0

                test_board.push(move)
                new_seq = move_seq + [move]
                new_cost = cost_so_far + move_cost

                if test_board.board_fen() == target_board_fen and new_cost < best_cost:
                    best_cost = new_cost
                    best_seq = new_seq

                next_queue.append((test_board, new_seq, new_cost))

        queue = sorted(next_queue, key=lambda x: x[2])[:beam_width]

        if best_seq is not None:
            return best_seq

    return None

In [60]:
class AdaptiveThreshold:
    def __init__(
        self,
        window_size=45,
        warmup_frames=10,
        start_k=5.0,
        stop_k=2.0,
        min_noise=1e-4
    ):
        self.values = deque(maxlen=window_size)
        self.warmup_frames = warmup_frames
        self.start_k = start_k
        self.stop_k = stop_k
        self.min_noise = min_noise

    def ready(self):
        return len(self.values) >= self.warmup_frames

    def update(self, value):
        self.values.append(float(value))

    def stats(self):
        arr = np.array(self.values, dtype=np.float32)
        median = np.median(arr)
        mad = np.median(np.abs(arr - median))
        noise = max(1.4826 * mad, self.min_noise)
        return median, noise

    def moving_threshold(self):
        median, noise = self.stats()
        return median + self.start_k * noise

    def stable_threshold(self):
        median, noise = self.stats()
        return median + self.stop_k * noise


def board_frame_difference(prev_board, curr_board):
    prev_gray = cv2.cvtColor(prev_board, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_board, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(prev_gray, curr_gray)
    return float(np.mean(diff)), diff


def board_similarity_diff(board_a, board_b):
    gray_a = cv2.cvtColor(board_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(board_b, cv2.COLOR_BGR2GRAY)
    return float(np.mean(cv2.absdiff(gray_a, gray_b)))


def get_square_name(row, col, flipped=False):
    files = "abcdefgh" if not flipped else "hgfedcba"
    ranks = "87654321" if not flipped else "12345678"
    return files[col] + ranks[row]


def square_to_row_col(square, flipped=False):
    files = "abcdefgh" if not flipped else "hgfedcba"
    ranks = "87654321" if not flipped else "12345678"
    return ranks.index(square[1]), files.index(square[0])


def square_difference_scores(prev_board, curr_board, board_size=800, flipped=False, inner_margin=15):
    cell_size = board_size // 8
    scores = {}

    prev_gray = cv2.cvtColor(prev_board, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_board, cv2.COLOR_BGR2GRAY)

    for r in range(8):
        for c in range(8):
            y1, y2 = r * cell_size, (r + 1) * cell_size
            x1, x2 = c * cell_size, (c + 1) * cell_size

            diff = cv2.absdiff(prev_gray[y1:y2, x1:x2], curr_gray[y1:y2, x1:x2])

            if inner_margin > 0:
                diff = diff[inner_margin:-inner_margin, inner_margin:-inner_margin]

            scores[get_square_name(r, c, flipped)] = float(np.mean(diff))

    return scores


def get_changed_squares(prev_board, curr_board, board_size=800, flipped=False, top_k=4):
    scores = square_difference_scores(prev_board, curr_board, board_size, flipped)
    changed = sorted(scores.keys(), key=lambda sq: scores[sq], reverse=True)
    return changed[:top_k], scores


def draw_changed_squares(board_img, changed, board_size=800, flipped=False):
    vis = board_img.copy()
    cell_size = board_size // 8

    for square in changed:
        row, col = square_to_row_col(square, flipped=flipped)

        x1 = col * cell_size
        y1 = row * cell_size
        x2 = x1 + cell_size
        y2 = y1 + cell_size

        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 3)
        cv2.putText(vis, square, (x1 + 8, y1 + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    return vis


def render_simple_chess_board(board, size=800, flipped=False, tint=None):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    cell = size // 8

    light = np.array([240, 217, 181], dtype=np.uint8)
    dark = np.array([181, 136, 99], dtype=np.uint8)

    if tint is not None:
        tint_arr = np.array(tint, dtype=np.uint8)
        light = (0.78 * light + 0.22 * tint_arr).astype(np.uint8)
        dark = (0.78 * dark + 0.22 * tint_arr).astype(np.uint8)

    for row in range(8):
        for col in range(8):
            color = light if (row + col) % 2 == 0 else dark
            x1, y1 = col * cell, row * cell
            cv2.rectangle(img, (x1, y1), (x1 + cell, y1 + cell), color.tolist(), -1)

            square = chess.square(7 - col, row) if flipped else chess.square(col, 7 - row)
            piece = board.piece_at(square)

            if piece is not None:
                text = piece.symbol()
                text_color = (20, 20, 20) if piece.color == chess.BLACK else (245, 245, 245)
                outline_color = (245, 245, 245) if piece.color == chess.BLACK else (20, 20, 20)

                font = cv2.FONT_HERSHEY_SIMPLEX
                scale = 1.8
                thickness = 4

                (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
                tx = x1 + (cell - tw) // 2
                ty = y1 + (cell + th) // 2

                cv2.putText(img, text, (tx, ty), font, scale, outline_color, thickness + 3)
                cv2.putText(img, text, (tx, ty), font, scale, text_color, thickness)

    return img


def add_panel_title(img, title, title_height=36, color=(30, 30, 30)):
    h, w = img.shape[:2]
    title_bar = np.zeros((title_height, w, 3), dtype=np.uint8)
    title_bar[:] = color

    cv2.putText(title_bar, title, (10, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

    return np.vstack([title_bar, img])


def board_is_cnn_ready(
    board_img,
    reference_board=None,
    board_size=800,
    max_active_squares=10,
    active_fraction=0.25
):
    if board_img is None or board_img.size == 0:
        return False, "bad_board_image", None

    if reference_board is None:
        return True, "cnn_ready_no_reference", None

    _, scores = get_changed_squares(reference_board, board_img, board_size=board_size, top_k=8)

    if scores is None or len(scores) == 0:
        return True, "cnn_ready_no_scores", scores

    max_score = max(scores.values())

    if max_score <= 0:
        return True, "cnn_ready_no_motion", scores

    active_squares = [
        sq for sq, score in scores.items()
        if score > active_fraction * max_score
    ]

    if len(active_squares) > max_active_squares:
        return False, f"too_many_active_squares_{len(active_squares)}", scores

    return True, "cnn_ready", scores

In [61]:
class ChessVideoRecoveryTracker:
    def __init__(self, config=None):
        self.config = config if config is not None else TrackerConfig()

        self.state = "NO_BOARD"
        self.frame_idx = 0

        self.board_box = None
        self.no_board_count = 0

        self.prev_raw_board = None
        self.last_stable_board = None
        self.movement_start_board = None

        self.moving = False
        self.stable_count = 0
        self.candidate_stable_board = None
        self.confirm_stable_count = 0

        self.periodic_validation_frames = max(1, int(self.config.periodic_validation_seconds * 30))
        self.last_periodic_cnn_frame = -10**9
        self.last_motion_frame = -10**9

        self.adaptive = AdaptiveThreshold(
            window_size=self.config.adaptive_window_size,
            warmup_frames=self.config.adaptive_warmup_frames,
            start_k=self.config.adaptive_start_k,
            stop_k=self.config.adaptive_stop_k
        )

        self.game = create_new_game_state()

        self.prospective_board = self.game["board"].copy()
        self.prospective_move_list = []
        self.pending_prospective_moves = []
        self.pending_prospective_sans = []
        self.pending_prospective_fen = None

        self.last_cnn_fen = None
        self.last_cnn_reason = "none"
        self.last_changed_squares = []
        self.periodic_cnn_ready_flash = 0

        self.failed_stitch_count = 0
        self.disable_recovery_request = False

        self.checkpoints = []
        self.last_checkpoint = None

        self.recovery_requested = False
        self.recovery_reason = ""
        self.recovery_target_fen = None
        self.recovery_failed_frame = None

        self.recovery_mode = "NORMAL"
        self.recovery_attempt = 0
        self.recovery_message = ""

        self.diff_history = []
        self.derivative_history = []
        self.moving_threshold_history = []
        self.stable_threshold_history = []
        self.median_history = []
        self.noise_history = []
        self.event_history = []

    def save_checkpoint(self, reason="synced"):
        checkpoint = {
            "frame_idx": self.frame_idx,
            "board": self.game["board"].copy(),
            "move_list": self.game["move_list"].copy(),
            "fen_sequence": self.game["fen_sequence"].copy(),
            "last_accepted_fen": self.game["last_accepted_fen"],
            "prospective_board": self.prospective_board.copy(),
            "prospective_move_list": self.prospective_move_list.copy(),
            "reason": reason
        }

        self.last_checkpoint = checkpoint
        self.checkpoints.append(checkpoint)

        if len(self.checkpoints) > 30:
            self.checkpoints.pop(0)

    def restore_checkpoint(self, checkpoint):
        self.game["board"] = checkpoint["board"].copy()
        self.game["move_list"] = checkpoint["move_list"].copy()
        self.game["fen_sequence"] = checkpoint["fen_sequence"].copy()
        self.game["last_accepted_fen"] = checkpoint["last_accepted_fen"]

        self.prospective_board = checkpoint["prospective_board"].copy()
        self.prospective_move_list = checkpoint["prospective_move_list"].copy()

        self.pending_prospective_moves = []
        self.pending_prospective_sans = []
        self.pending_prospective_fen = None

        self.failed_stitch_count = 0
        self.recovery_requested = False
        self.recovery_reason = ""
        self.recovery_target_fen = None

    def detect_or_update_board(self, frame):
        should_detect = (
            self.board_box is None or
            self.frame_idx % self.config.board_detect_every_n_frames == 0
        )

        if not should_detect:
            return self.board_box is not None

        result = detect_board_contour_grid_2d(frame, debug=False)

        if result is None:
            self.no_board_count += 1

            if self.no_board_count > self.config.reset_if_board_missing_frames:
                self.state = "NO_BOARD"
                self.board_box = None
                self.reset_motion_only()

            return False

        self.board_box = result["final_box"]
        self.no_board_count = 0

        if self.state == "NO_BOARD":
            self.state = "BOARD_FOUND"

        return True

    def reset_motion_only(self):
        self.prev_raw_board = None
        self.last_stable_board = None
        self.movement_start_board = None
        self.moving = False
        self.stable_count = 0
        self.candidate_stable_board = None
        self.confirm_stable_count = 0
        self.last_motion_frame = -10**9

        self.adaptive = AdaptiveThreshold(
            window_size=self.config.adaptive_window_size,
            warmup_frames=self.config.adaptive_warmup_frames,
            start_k=self.config.adaptive_start_k,
            stop_k=self.config.adaptive_stop_k
        )

    def process_frame(self, frame):
        self.frame_idx += 1

        if not self.detect_or_update_board(frame):
            return self.draw_dashboard(frame, None, "No board found")

        board_ready = crop_and_prepare_board(
            frame,
            self.board_box,
            output_size=self.config.output_size
        )

        if board_ready is None:
            return self.draw_dashboard(frame, None, "Board crop failed")

        self.update_motion_state(board_ready)

        if self.moving:
            status_text = "Movement detected"
        elif self.state == "TRACKING_GAME":
            status_text = "Tracking game"
        elif self.state == "BOARD_FOUND":
            status_text = "Board found / warming up"
        else:
            status_text = self.state

        return self.draw_dashboard(frame, board_ready, status_text)

    def update_motion_state(self, board_ready):
        if self.prev_raw_board is None:
            self.prev_raw_board = board_ready.copy()
            self.last_stable_board = board_ready.copy()
            self.state = "BOARD_FOUND"
            return

        diff_score, _ = board_frame_difference(self.prev_raw_board, board_ready)
        prev_diff = self.diff_history[-1] if len(self.diff_history) > 0 else diff_score
        derivative = diff_score - prev_diff

        self.diff_history.append(diff_score)
        self.derivative_history.append(derivative)

        if not self.adaptive.ready():
            self.adaptive.update(diff_score)
            self.moving_threshold_history.append(None)
            self.stable_threshold_history.append(None)
            self.median_history.append(None)
            self.noise_history.append(None)
            self.event_history.append("warmup")
            self.prev_raw_board = board_ready.copy()
            self.last_stable_board = board_ready.copy()
            return

        median, noise = self.adaptive.stats()
        moving_threshold = self.adaptive.moving_threshold()
        stable_threshold = self.adaptive.stable_threshold()

        self.moving_threshold_history.append(moving_threshold)
        self.stable_threshold_history.append(stable_threshold)
        self.median_history.append(median)
        self.noise_history.append(noise)

        if diff_score > moving_threshold and not self.moving:
            self.moving = True
            self.state = "MOVEMENT_ACTIVE"

            self.last_motion_frame = self.frame_idx
            self.stable_count = 0
            self.candidate_stable_board = None
            self.confirm_stable_count = 0
            self.movement_start_board = self.last_stable_board.copy()

            self.event_history.append("movement_start")

            if self.config.show_debug:
                print(f"\nFrame {self.frame_idx}: movement started")
                print("diff_score:", diff_score)
                print("moving_threshold:", moving_threshold)
                print("derivative:", derivative)

        elif self.moving:
            self.event_history.append("moving")
            self.handle_moving_frame(board_ready, diff_score, stable_threshold)

        else:
            self.event_history.append("stable")
            self.adaptive.update(diff_score)
            self.last_stable_board = board_ready.copy()
            self.state = "TRACKING_GAME"

            frames_since_periodic = self.frame_idx - self.last_periodic_cnn_frame
            frames_since_motion = self.frame_idx - self.last_motion_frame

            if (
                frames_since_periodic >= self.periodic_validation_frames
                and frames_since_motion >= self.config.periodic_cooldown_after_motion
                and not self.moving
            ):
                ready, reason, _ = board_is_cnn_ready(
                    board_ready,
                    reference_board=self.last_stable_board,
                    board_size=self.config.output_size,
                    max_active_squares=self.config.unstable_active_square_limit,
                    active_fraction=self.config.unstable_score_fraction
                )

                self.last_cnn_reason = reason

                if ready:
                    self.last_periodic_cnn_frame = self.frame_idx
                    self.periodic_cnn_ready_flash = 8

                    if self.config.show_debug:
                        print(f"\nFrame {self.frame_idx}: periodic stable CNN check")

                    self.handle_new_stable_board(board_ready.copy(), source="periodic")

        self.prev_raw_board = board_ready.copy()

        if self.periodic_cnn_ready_flash > 0:
            self.periodic_cnn_ready_flash -= 1

    def handle_moving_frame(self, board_ready, diff_score, stable_threshold):
        if diff_score < stable_threshold:
            self.stable_count += 1
        else:
            self.stable_count = 0
            self.candidate_stable_board = None
            self.confirm_stable_count = 0

        if self.stable_count < self.config.required_stable_frames:
            return

        if self.candidate_stable_board is None:
            self.candidate_stable_board = board_ready.copy()
            self.confirm_stable_count = 0

            if self.config.show_debug:
                print(f"Frame {self.frame_idx}: candidate stable board found")

            return

        confirm_diff = board_similarity_diff(self.candidate_stable_board, board_ready)

        if confirm_diff < self.config.confirm_board_diff_threshold:
            self.confirm_stable_count += 1
        else:
            self.candidate_stable_board = board_ready.copy()
            self.confirm_stable_count = 0

        if self.confirm_stable_count >= self.config.required_confirm_frames:
            new_stable_board = self.candidate_stable_board.copy()

            self.moving = False
            self.stable_count = 0
            self.candidate_stable_board = None
            self.confirm_stable_count = 0

            self.handle_new_stable_board(new_stable_board, source="trigger")

            self.last_stable_board = new_stable_board.copy()
            self.state = "TRACKING_GAME"

    def handle_new_stable_board(self, new_stable_board, source="trigger"):
        if self.movement_start_board is not None:
            changed, scores = get_changed_squares(
                self.movement_start_board,
                new_stable_board,
                board_size=self.config.output_size,
                flipped=False,
                top_k=8
            )
            self.last_changed_squares = changed
        else:
            scores = None
            self.last_changed_squares = []

        if source == "periodic":
            ready, reason, readiness_scores = board_is_cnn_ready(
                new_stable_board,
                reference_board=self.last_stable_board,
                board_size=self.config.output_size,
                max_active_squares=self.config.unstable_active_square_limit,
                active_fraction=self.config.unstable_score_fraction
            )

            self.last_cnn_reason = reason

            if not ready:
                if self.config.show_debug:
                    print(f"Rejected periodic CNN frame: {reason}")
                return

            if readiness_scores is not None:
                scores = readiness_scores

        else:
            self.last_cnn_reason = "triggered_cnn"

        self.periodic_cnn_ready_flash = 8

        if self.config.show_debug:
            print(f"\nFrame {self.frame_idx}: sending board to CNN | source={source}")

        cnn_board_fen = safe_extract_fen_from_board_image(
            new_stable_board,
            verbose=self.config.show_debug
        )

        if cnn_board_fen is None:
            return

        self.last_cnn_fen = cnn_board_fen

        if self.config.show_debug:
            print("CNN board FEN:", cnn_board_fen)
            print("Definitive board FEN:", self.game["board"].board_fen())

        if should_reset_for_new_game(cnn_board_fen, self.game["move_list"]):
            self.__init__(self.config)
            return

        self.process_cnn_fen_prospectively(cnn_board_fen, scores=scores, source=source)

    def commit_pending_prospective_move(self):
        if len(self.pending_prospective_moves) == 0:
            return

        for move in self.pending_prospective_moves:
            san = self.game["board"].san(move)
            self.game["board"].push(move)
            self.game["move_list"].append(san)
            self.game["last_accepted_fen"] = self.game["board"].board_fen()
            self.game["fen_sequence"].append(self.game["last_accepted_fen"])
            self.game["game_started"] = True

            if self.config.show_debug:
                print("Committed definitive move:", san)

        self.pending_prospective_moves = []
        self.pending_prospective_sans = []
        self.pending_prospective_fen = None
        self.failed_stitch_count = 0

        self.save_checkpoint(reason="confirmed_prospective")

    def process_cnn_fen_prospectively(self, cnn_board_fen, scores=None, source="trigger"):
        definitive_fen = self.game["board"].board_fen()
        prospective_fen = self.prospective_board.board_fen()

        if cnn_board_fen == definitive_fen and len(self.pending_prospective_moves) == 0:
            self.save_checkpoint(reason="cnn_matches_definitive")
            if self.config.show_debug:
                print("CNN matches definitive board. No move needed.")
            return

        if len(self.pending_prospective_moves) > 0:
            if cnn_board_fen == prospective_fen:
                self.commit_pending_prospective_move()
                return

            followup_moves = stitch_moves_from_fen(
                self.prospective_board,
                cnn_board_fen,
                max_depth=self.config.max_stitch_depth,
                scores=scores
            )

            if followup_moves is not None:
                self.commit_pending_prospective_move()
                self.add_new_prospective_moves(
                    self.game["board"],
                    cnn_board_fen,
                    scores=scores,
                    source=source
                )
                return

            if self.config.show_debug:
                print("Pending prospective move did not persist. Replacing prospective candidate.")

            self.prospective_board = self.game["board"].copy()
            self.prospective_move_list = self.game["move_list"].copy()
            self.pending_prospective_moves = []
            self.pending_prospective_sans = []
            self.pending_prospective_fen = None

        self.add_new_prospective_moves(
            self.game["board"],
            cnn_board_fen,
            scores=scores,
            source=source
        )

    def add_new_prospective_moves(self, base_board, target_board_fen, scores=None, source="trigger"):
        if base_board.board_fen() == target_board_fen:
            return

        move = stitch_one_move_from_fen(base_board, target_board_fen, scores=scores)
        move_distance = 0

        if move is None and self.config.use_fuzzy_single_move:
            move, move_distance = stitch_one_move_from_fen_fuzzy(
                base_board,
                target_board_fen,
                scores=scores,
                max_distance=self.config.fuzzy_max_distance
            )

        if move is not None:
            temp_board = base_board.copy()
            san = temp_board.san(move)
            temp_board.push(move)

            self.prospective_board = temp_board
            self.prospective_move_list = self.game["move_list"] + [san]
            self.pending_prospective_moves = [move]
            self.pending_prospective_sans = [san]
            self.pending_prospective_fen = temp_board.board_fen()

            if self.config.show_debug:
                if move_distance == 0:
                    print(f"Prospective move from {source}: {san}")
                else:
                    print(f"Prospective fuzzy move from {source}: {san} | distance={move_distance}")
            return

        stitched_moves = None

        if self.config.allow_multimove_stitching:
            stitched_moves = stitch_moves_from_fen(
                base_board,
                target_board_fen,
                max_depth=self.config.max_stitch_depth,
                scores=scores
            )

        if stitched_moves is not None:
            temp_board = base_board.copy()
            sans = []

            for m in stitched_moves:
                san = temp_board.san(m)
                temp_board.push(m)
                sans.append(san)

            self.prospective_board = temp_board
            self.prospective_move_list = self.game["move_list"] + sans
            self.pending_prospective_moves = stitched_moves
            self.pending_prospective_sans = sans
            self.pending_prospective_fen = temp_board.board_fen()

            if self.config.show_debug:
                print("Candidate multi-move:", sans)
                print("Target FEN:", target_board_fen)
                print("Result FEN:", temp_board.board_fen())
            return

        self.failed_stitch_count += 1

        if self.config.show_debug:
            print(f"Rejected CNN FEN: could not stitch prospectively ({self.failed_stitch_count})")

        if (
            self.failed_stitch_count >= self.config.failed_stitch_recovery_threshold
            and not self.disable_recovery_request
        ):
            self.recovery_requested = True
            self.recovery_reason = "failed_stitch_count"
            self.recovery_target_fen = target_board_fen
            self.recovery_failed_frame = self.frame_idx

    def draw_dashboard(self, frame, board_ready, status_text=""):
        panel_size = self.config.panel_size
        output_size = self.config.output_size

        if board_ready is None:
            board_panel = np.zeros((panel_size, panel_size, 3), dtype=np.uint8)
            cv2.putText(board_panel, "No board", (70, panel_size // 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        else:
            board_panel_full = board_ready.copy()

            if self.periodic_cnn_ready_flash > 0:
                cv2.rectangle(board_panel_full, (8, 8), (output_size - 8, output_size - 8),
                              (0, 255, 255), 6)
                cv2.putText(board_panel_full, "CNN CHECK", (25, 45),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)

            if self.config.show_highlights and self.moving and self.movement_start_board is not None:
                changed, _ = get_changed_squares(
                    self.movement_start_board,
                    board_ready,
                    board_size=output_size,
                    flipped=False,
                    top_k=4
                )
                board_panel_full = draw_changed_squares(board_panel_full, changed, board_size=output_size)

            board_panel = cv2.resize(board_panel_full, (panel_size, panel_size))

        definitive_board = render_simple_chess_board(
            self.game["board"],
            size=output_size,
            tint=(60, 120, 255)
        )
        definitive_board = cv2.resize(definitive_board, (panel_size, panel_size))

        prospective_board = render_simple_chess_board(
            self.prospective_board,
            size=output_size,
            tint=(60, 220, 120)
        )
        prospective_board = cv2.resize(prospective_board, (panel_size, panel_size))

        info_panel = self.make_info_panel(panel_size, status_text)

        board_panel = add_panel_title(board_panel, "Extracted Board", color=(30, 30, 30))
        definitive_board = add_panel_title(definitive_board, "Definitive Board - BLUE", color=(80, 70, 20))
        prospective_board = add_panel_title(prospective_board, "Prospective Board - GREEN", color=(25, 90, 35))
        info_panel = add_panel_title(info_panel, "Stats / Recovery", color=(30, 30, 30))

        top = np.hstack([board_panel, definitive_board])
        bottom = np.hstack([prospective_board, info_panel])

        return np.vstack([top, bottom])

    def make_info_panel(self, size, status_text):
        panel = np.zeros((size, size, 3), dtype=np.uint8)
        y = 28

        def put(text, scale=0.40, thickness=1, dy=20, color=(255, 255, 255)):
            nonlocal y
            cv2.putText(panel, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX,
                        scale, color, thickness)
            y += dy

        mode_color = (255, 255, 255) if self.recovery_mode == "NORMAL" else (0, 0, 255)

        put(f"Status: {status_text}", scale=0.42, dy=23)
        put(f"Frame: {self.frame_idx}", scale=0.40, dy=20)
        put(f"Mode: {self.recovery_mode}", scale=0.42, thickness=2, dy=22, color=mode_color)
        put(f"Recovery try: {self.recovery_attempt}", scale=0.38, dy=20)
        put(self.recovery_message[:34], scale=0.34, dy=20)
        put(f"CNN: {self.last_cnn_reason}", scale=0.36, dy=20)
        put(f"Failed stitches: {self.failed_stitch_count}", scale=0.38, dy=22)

        if self.periodic_cnn_ready_flash > 0:
            put("CNN CHECK ACTIVE", scale=0.40, thickness=2, dy=24, color=(0, 255, 255))
        else:
            put("CNN idle", scale=0.38, dy=22)

        put(f"Def moves: {len(self.game['move_list'])}", scale=0.40, dy=20)
        put(f"Pros moves: {len(self.prospective_move_list)}", scale=0.40, dy=20)

        pending = " ".join(self.pending_prospective_sans)
        if pending == "":
            pending = "None"

        put(f"Pending: {pending[:25]}", scale=0.34, dy=22)

        put("Definitive:", scale=0.40, dy=20)
        recent = self.game["move_list"][-10:]
        line = ""

        for i, move in enumerate(recent):
            move_num = len(self.game["move_list"]) - len(recent) + i + 1
            token = f"{move_num}.{move} "

            if len(line + token) > 30:
                put(line, scale=0.33, dy=17)
                line = token
            else:
                line += token

        if line:
            put(line, scale=0.33, dy=20)

        put("Prospective:", scale=0.40, dy=20)
        recent = self.prospective_move_list[-10:]
        line = ""

        for i, move in enumerate(recent):
            move_num = len(self.prospective_move_list) - len(recent) + i + 1
            token = f"{move_num}.{move} "

            if len(line + token) > 30:
                put(line, scale=0.33, dy=17)
                line = token
            else:
                line += token

        if line:
            put(line, scale=0.33, dy=17)

        return panel
    

def update_winner_prediction_for_tracker(tracker):
    if not hasattr(tracker, "last_prediction_move_count"):
        tracker.last_prediction_move_count = -1
        tracker.last_winner_prediction = None
        tracker.winner_prediction_history = []

    move_list = tracker.game["move_list"]
    move_count = len(move_list)

    if move_count == 0:
        return

    if move_count == tracker.last_prediction_move_count:
        return

    try:
        pred = predict_winner_from_move_list(move_list)

        tracker.last_winner_prediction = pred
        tracker.last_prediction_move_count = move_count

        tracker.winner_prediction_history.append({
            "move_count": move_count,
            "classes": pred["classes"],
            "rf_probs": pred["rf_probs"],
            "transformer_probs": pred["transformer_probs"],
            "avg_probs": pred["avg_probs"],
        })

        tracker.winner_prediction_history = tracker.winner_prediction_history[-100:]

    except Exception as e:
        print("Winner prediction failed:", e)
        tracker.last_winner_prediction = None


def draw_probability_bars(panel, pred, x, y, width=320, bar_h=13):
    if pred is None:
        cv2.putText(panel, "Winner prediction: waiting...", (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1)
        return y + 22

    classes = pred["classes"]

    rows = [
        ("RF", pred["rf_probs"]),
        ("Trans", pred["transformer_probs"]),
        ("Avg", pred["avg_probs"]),
    ]

    colors = {
        "white": (245, 245, 245),
        "black": (90, 90, 90),
        "draw": (0, 255, 255),
    }

    for model_name, probs in rows:
        cv2.putText(panel, model_name, (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)

        x0 = x + 55

        for cls, p in zip(classes, probs):
            p = float(p)
            bar_w = int(width * p)

            color = colors.get(str(cls).lower(), (180, 180, 255))

            cv2.rectangle(panel, (x0, y - 10), (x0 + bar_w, y + bar_h - 10), color, -1)
            cv2.rectangle(panel, (x0, y - 10), (x0 + width, y + bar_h - 10), (100, 100, 100), 1)

            label = f"{cls}: {100*p:.1f}%"
            cv2.putText(panel, label, (x0 + width + 8, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.32, color, 1)

            y += 16

        y += 6

    best_idx = int(np.argmax(pred["avg_probs"]))
    best_label = classes[best_idx]
    best_prob = float(pred["avg_probs"][best_idx])

    cv2.putText(panel, f"Best: {best_label} ({100*best_prob:.1f}%)",
                (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 255, 255), 1)

    return y + 24


def draw_winner_probability_plot(panel, tracker, x, y, width=340, height=95):
    history = getattr(tracker, "winner_prediction_history", [])

    cv2.rectangle(panel, (x, y), (x + width, y + height), (100, 100, 100), 1)

    cv2.putText(panel, "Winner prob over moves", (x, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.34, (220, 220, 220), 1)

    if len(history) < 2:
        return

    classes = history[-1]["classes"]

    colors = {
        "white": (245, 245, 245),
        "black": (90, 90, 90),
        "draw": (0, 255, 255),
    }

    plot_history = history[-40:]
    xs = np.linspace(x + 4, x + width - 4, len(plot_history)).astype(int)

    for class_idx, cls in enumerate(classes):
        pts = []

        for px, item in zip(xs, plot_history):
            p = float(item["avg_probs"][class_idx])
            py = int((y + height - 4) - p * (height - 8))
            pts.append((px, py))

        color = colors.get(str(cls).lower(), (180, 180, 255))

        for i in range(len(pts) - 1):
            cv2.line(panel, pts[i], pts[i + 1], color, 2)

        cv2.putText(panel, str(cls), (pts[-1][0] - 34, pts[-1][1] - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)


# Save original make_info_panel once
if not hasattr(ChessVideoRecoveryTracker, "_original_make_info_panel"):
    ChessVideoRecoveryTracker._original_make_info_panel = ChessVideoRecoveryTracker.make_info_panel


def make_info_panel_with_predictions(self, size, status_text):
    update_winner_prediction_for_tracker(self)

    panel = ChessVideoRecoveryTracker._original_make_info_panel(self, size, status_text)

    # Cover lower portion of the info panel so prediction display is readable
    start_y = int(size * 0.56)
    cv2.rectangle(panel, (0, start_y), (size, size), (0, 0, 0), -1)

    y = start_y + 22

    pred = getattr(self, "last_winner_prediction", None)

    y = draw_probability_bars(
        panel,
        pred,
        x=10,
        y=y,
        width=115,
        bar_h=11
    )

    draw_winner_probability_plot(
        panel,
        self,
        x=10,
        y=min(y + 8, size - 105),
        width=size - 20,
        height=90
    )

    return panel


ChessVideoRecoveryTracker.make_info_panel = make_info_panel_with_predictions

print("Winner prediction dashboard enabled.")

Winner prediction dashboard enabled.


In [62]:
BASE_RECOVERY_CONFIGS = [
    {
        "name": "RECOVERY EASY",
        "required_stable_frames": 1,
        "required_confirm_frames": 1,
        "confirm_board_diff_threshold": 8.0,
        "periodic_cooldown_after_motion": 18,
        "max_stitch_depth": 5,
        "adaptive_start_k": 3.5,
        "adaptive_stop_k": 1.5,
    },
    {
        "name": "RECOVERY EASIER",
        "required_stable_frames": 1,
        "required_confirm_frames": 0,
        "confirm_board_diff_threshold": 12.0,
        "periodic_cooldown_after_motion": 20,
        "max_stitch_depth": 6,
        "adaptive_start_k": 2.5,
        "adaptive_stop_k": 1.2,
    },
    {
        "name": "RECOVERY AGGRESSIVE",
        "required_stable_frames": 1,
        "required_confirm_frames": 0,
        "confirm_board_diff_threshold": 18.0,
        "periodic_cooldown_after_motion": 24,
        "max_stitch_depth": 7,
        "adaptive_start_k": 1.8,
        "adaptive_stop_k": 1.0,
    }
]

RECOVERY_PERIODS = [ 0.85, 0.65, 0.95, 0.40, 1.05, 0.7, 0.8 ]

RECOVERY_CONFIGS = []

for base in BASE_RECOVERY_CONFIGS:
    for period in RECOVERY_PERIODS:
        cfg = base.copy()
        cfg["periodic_validation_seconds"] = period
        cfg["name"] = f"{base['name']} | period={period:.2f}s"
        RECOVERY_CONFIGS.append(cfg)


class VideoFrameSource:
    def __init__(self, video_path):
        self.cap = cv2.VideoCapture(video_path)

        if not self.cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")

        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        if self.fps <= 0:
            self.fps = 30

    def read(self):
        ret, frame = self.cap.read()
        if not ret:
            return None
        return frame

    def release(self):
        self.cap.release()


def load_video_frames(video_path, max_frames=None):
    source = VideoFrameSource(video_path)
    frames = []

    while True:
        frame = source.read()

        if frame is None:
            break

        frames.append(frame)

        if max_frames is not None and len(frames) >= max_frames:
            break

    fps = source.fps
    source.release()

    return frames, fps


def make_tracker_config(
    output_size=800,
    panel_size=380,
    show_debug=True,
    overrides=None
):
    kwargs = dict(
        output_size=output_size,
        panel_size=panel_size,
        show_debug=show_debug,

        required_stable_frames=2,
        required_confirm_frames=1,
        confirm_board_diff_threshold=4.0,

        periodic_validation_seconds=0.75,
        periodic_cooldown_after_motion=16,

        use_fuzzy_single_move=True,
        fuzzy_max_distance=1,
        allow_multimove_stitching=True,
        max_stitch_depth=4,

        unstable_active_square_limit=12,
        unstable_score_fraction=0.25,

        adaptive_start_k=5.0,
        adaptive_stop_k=2.0,
    )

    if overrides is not None:
        clean = overrides.copy()
        clean.pop("name", None)
        kwargs.update(clean)

    return TrackerConfig(**kwargs)


def draw_banner(dashboard, text, color=(0, 0, 180)):
    out = dashboard.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 54), color, -1)
    cv2.putText(out, text, (18, 36), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255, 255, 255), 2)
    return out


def make_rewind_dashboard(base_tracker, frame, frame_idx, checkpoint_frame, dashboard_size):
    tracker = base_tracker
    dash = tracker.draw_dashboard(frame, None, status_text="REWINDING")
    dash = cv2.resize(dash, dashboard_size)

    text = f"REWINDING: frame {frame_idx} back to checkpoint {checkpoint_frame}"
    dash = draw_banner(dash, text, color=(180, 60, 0))
    return dash


def replay_from_checkpoint(
    frames,
    checkpoint,
    start_frame_idx,
    end_frame_idx,
    target_fen,
    recovery_overrides,
    attempt_num,
    output_size=800,
    panel_size=380,
    show_debug=True,
    writer=None,
    dashboard_size=None,
    display=True,
    replay_delay_ms=15
):
    config = make_tracker_config(
        output_size=output_size,
        panel_size=panel_size,
        show_debug=show_debug,
        overrides=recovery_overrides
    )

    tracker = ChessVideoRecoveryTracker(config=config)
    tracker.restore_checkpoint(checkpoint)
    tracker.disable_recovery_request = True

    tracker.recovery_mode = recovery_overrides["name"]
    tracker.recovery_attempt = attempt_num
    tracker.recovery_message = f"Replay {start_frame_idx}->{end_frame_idx}"

    tracker.frame_idx = start_frame_idx

    for i in range(start_frame_idx, min(end_frame_idx + 1, len(frames))):
        dash = tracker.process_frame(frames[i])

        if dashboard_size is not None:
            dash = cv2.resize(dash, dashboard_size)

        dash = draw_banner(
            dash,
            f"RECOVERY REPLAY attempt {attempt_num}: {recovery_overrides['name']} | frame {i}",
            color=(0, 0, 180)
        )

        if writer is not None:
            writer.write(dash.astype(np.uint8))

        if display:
            cv2.imshow("Chess Recovery Tracker", dash)
            key = cv2.waitKey(replay_delay_ms) & 0xFF
            if key == ord("q"):
                break

    success = (
        tracker.game["board"].board_fen() == target_fen or
        tracker.prospective_board.board_fen() == target_fen
    )

    return success, tracker

def score_recovery_result(tracker, target_fen, start_move_count):
    score = 0

    current_fen = tracker.game["board"].board_fen()
    prospective_fen = tracker.prospective_board.board_fen()

    if current_fen == target_fen:
        score += 1000
    elif prospective_fen == target_fen:
        score += 850
    else:
        score -= 100 * board_fen_square_distance(current_fen, target_fen)

    new_moves = tracker.game["move_list"][start_move_count:]
    score -= 8 * len(new_moves)

    suspicious_count = 0
    for move in new_moves:
        if move.startswith("R") or move.startswith("K"):
            suspicious_count += 1
        if "+" in move:
            score += 2
        if "x" in move:
            score += 1

    score -= 15 * suspicious_count
    score -= 20 * tracker.failed_stitch_count

    return score


def run_chess_video_tracker_with_recovery(
    video_path,
    output_path=None,
    output_size=800,
    panel_size=380,
    show_debug=True,
    max_frames=None,
    display=True,
    normal_delay_ms=1,
    rewind_delay_ms=8,
    replay_delay_ms=15,
    rewind_padding_frames=45,
    replay_forward_padding_frames=35,
    max_recoveries_per_key=2
):
    frames, fps = load_video_frames(video_path, max_frames=max_frames)

    config = make_tracker_config(
        output_size=output_size,
        panel_size=panel_size,
        show_debug=show_debug
    )

    tracker = ChessVideoRecoveryTracker(config=config)

    dashboard_size = (panel_size * 2, (panel_size + 36) * 2)

    writer = None
    if output_path is not None:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_path, fourcc, fps, dashboard_size)

    recovery_attempts_by_key = {}

    i = 0

    while i < len(frames):
        dash = tracker.process_frame(frames[i])
        dash = cv2.resize(dash, dashboard_size)

        if writer is not None:
            writer.write(dash.astype(np.uint8))

        if display:
            cv2.imshow("Chess Recovery Tracker", dash)
            key = cv2.waitKey(normal_delay_ms) & 0xFF
            if key == ord("q"):
                break

        if tracker.recovery_requested and tracker.last_checkpoint is not None:
            failed_frame = i
            target_fen = tracker.recovery_target_fen
            checkpoint = tracker.last_checkpoint

            start_frame = max(0, checkpoint["frame_idx"] - rewind_padding_frames)
            end_frame = min(len(frames) - 1, failed_frame + replay_forward_padding_frames)

            recovery_key = (checkpoint["frame_idx"], failed_frame, target_fen)
            recovery_attempts_by_key[recovery_key] = recovery_attempts_by_key.get(recovery_key, 0) + 1

            if recovery_attempts_by_key[recovery_key] > max_recoveries_per_key:
                print("Skipping recovery: repeated same recovery loop.")
                tracker.recovery_requested = False
                tracker.failed_stitch_count = 0
                i += 1
                continue

            print("\n=== RECOVERY START ===")
            print("Reason:", tracker.recovery_reason)
            print("Checkpoint frame:", checkpoint["frame_idx"])
            print("Replay window:", start_frame, "to", end_frame)
            print("Target FEN:", target_fen)

            for j in range(failed_frame, start_frame, -1):
                rewind_dash = make_rewind_dashboard(
                    tracker,
                    frames[j],
                    frame_idx=j,
                    checkpoint_frame=checkpoint["frame_idx"],
                    dashboard_size=dashboard_size
                )

                if writer is not None:
                    writer.write(rewind_dash.astype(np.uint8))

                if display:
                    cv2.imshow("Chess Recovery Tracker", rewind_dash)
                    key = cv2.waitKey(rewind_delay_ms) & 0xFF
                    if key == ord("q"):
                        break

            best_recovery = None
            best_score = -10**9
            start_move_count = len(checkpoint["move_list"])

            EARLY_STOP_SCORE = 900
            MIN_SUCCESSFUL_ATTEMPTS_BEFORE_STOP = 2

            successful_attempts = 0

            for attempt_num, recovery_overrides in enumerate(RECOVERY_CONFIGS, start=1):
                success, recovery_tracker = replay_from_checkpoint(
                    frames=frames,
                    checkpoint=checkpoint,
                    start_frame_idx=start_frame,
                    end_frame_idx=end_frame,
                    target_fen=target_fen,
                    recovery_overrides=recovery_overrides,
                    attempt_num=attempt_num,
                    output_size=output_size,
                    panel_size=panel_size,
                    show_debug=show_debug,
                    writer=writer,
                    dashboard_size=dashboard_size,
                    display=display,
                    replay_delay_ms=replay_delay_ms
                )

                recovery_score = score_recovery_result(
                    recovery_tracker,
                    target_fen,
                    start_move_count=start_move_count
                )

                print(
                    f"Recovery attempt {attempt_num}: "
                    f"{recovery_overrides['name']} | "
                    f"success={success} | score={recovery_score}"
                )

                if success:
                    successful_attempts += 1

                    if recovery_score > best_score:
                        best_score = recovery_score
                        best_recovery = recovery_tracker

                if (
                    best_recovery is not None
                    and best_score >= EARLY_STOP_SCORE
                    and successful_attempts >= MIN_SUCCESSFUL_ATTEMPTS_BEFORE_STOP
                ):
                    print("Early stopping recovery search: strong recovery found.")
                    break

            if best_recovery is not None and best_score > 500:
                print("RECOVERY ACCEPTED")
                tracker = best_recovery
                tracker.disable_recovery_request = False
                tracker.recovery_mode = "NORMAL"
                tracker.recovery_message = "Recovered"
                tracker.recovery_requested = False
                tracker.failed_stitch_count = 0

                i = end_frame
            else:
                print("RECOVERY FAILED - continuing normal pass")
                tracker.recovery_requested = False
                tracker.failed_stitch_count = 0

            print("=== RECOVERY END ===\n")

        i += 1

    if writer is not None:
        writer.release()

    if display:
        cv2.destroyAllWindows()

    print("\nFinal move list:")
    print(tracker.game["move_list"])

    print("\nFinal FEN:")
    print(tracker.game["board"].fen())

    if output_path is not None:
        print("\nSaved:", output_path)

    return tracker

In [63]:
tracker = run_chess_video_tracker_with_recovery(
    video_path="videos/chess_recording_1.mp4",
    output_path="cnn_fen_tracker_recovery_clean.mp4",
    panel_size=380,
    output_size=800,
    show_debug=False,
    display=True,
    max_frames=None
)

C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\chessimg2pos\__init__.py:13: ResourceWarning: unclosed file <_io.BufferedReader name='C:\\Users\\brenn\\AppData\\Local\\Temp\\tmptzejaiqq.png'>
  result = predictor.predict_chessboard(image_path)
C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\chessimg2pos\__init__.py:13: ResourceWarning: unclosed file <_io.BufferedReader name='C:\\Users\\brenn\\AppData\\Local\\Temp\\tmpgu13iq6b.png'>
  result = predictor.predict_chessboard(image_path)
C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\chessimg2pos\__init__.py:13: ResourceWarning: unclosed file <_io.BufferedReader name='C:\\Users\\brenn\\AppData\\Local\\Temp\\tmpdlb1_cob.png'>
  result = predictor.predict_chessboard(image_path)
C:\Users\brenn\AppData\Roaming\Python\Python311\site-packages\chessimg2pos\__init__.py:13: ResourceWarning: unclosed file <_io.BufferedReader name='C:\\Users\\brenn\\AppData\\Local\\Temp\\tmpflh1s307.png'>
  result = predictor


Final move list:
['e4', 'e5', 'Nf3', 'd6', 'Bc4']

Final FEN:
rnbqkbnr/ppp2ppp/3p4/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 1 3

Saved: cnn_fen_tracker_recovery_clean.mp4
